# A Dense LLM from Scratch, Loaded with Real Weights

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davidwhiteboard/llm_from_scratch/blob/main/src/architecture/9.%20architecture.ipynb)

The previous notebooks each built one component. This one wires them into a complete
decoder-only model, then loads the published **Qwen3-0.6B** weights into it and checks the
output against the reference implementation. Matching logits verify every component at once:
a wrong RoPE layout, a wrong head grouping, or a misplaced normalization all produce noise.

$$\text{residual stream} \;\leftarrow\; \text{residual stream} + \text{sublayer}(\text{norm}(\text{residual stream}))$$

| Section | Component | Built in |
|---------|-----------|----------|
| §2 | RMSNorm, and QK-Norm on the head dim | normalization notebook |
| §3 | RoPE, split-half convention | positional encoding notebook |
| §4 | Grouped-Query Attention | attention variants notebook |
| §5 | SwiGLU feed-forward | activations notebook |
| §6-7 | The block, the edges, the parameter budget | here |
| §8-10 | Loading weights, verifying, generating | here |

Runs on a free Colab GPU, and on CPU if you are patient.

In [ ]:
# On Colab the first line installs what is missing; locally it is a no-op.
try:
    import google.colab  # noqa: F401
    !pip -q install "transformers>=4.51" safetensors huggingface_hub
except ImportError:
    pass

import math, os, json
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

MODEL_ID = "Qwen/Qwen3-0.6B"
CACHE_DIR = os.environ.get("LLM_FS_CACHE")     # None on Colab -> default HF cache
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
print("torch", torch.__version__, "| device", DEVICE)

## §1 - Configuration and the parameter budget

Every field below is read straight from the model's published `config.json`. The budget is
worth computing before any weights exist: it says where the parameters actually sit.

| Symbol | Value | Meaning |
|---|---|---|
| `d_model` | 1024 | width of the residual stream |
| `n_layers` | 28 | blocks stacked |
| `n_heads` / `n_kv_heads` | 16 / 8 | grouped-query ratio of 2 |
| `head_dim` | 128 | **not** `d_model / n_heads`; deliberately decoupled |
| `d_ff` | 3072 | hidden width of the feed-forward network |

> `16 x 128 = 2048`, twice the stream width. The query projection widens, the output projection narrows back.

In [ ]:
@dataclass
class Config:
    """Qwen3-0.6B, straight from its published config.json."""
    d_model: int = 1024
    n_layers: int = 28
    n_heads: int = 16
    n_kv_heads: int = 8            # 2 query heads per KV head
    head_dim: int = 128            # decoupled: 16 * 128 = 2048 != d_model
    d_ff: int = 3072
    vocab_size: int = 151936
    norm_eps: float = 1e-6
    rope_theta: float = 1_000_000.0
    tie_embeddings: bool = True

    @property
    def n_rep(self) -> int:
        """How many query heads share one KV head."""
        return self.n_heads // self.n_kv_heads


cfg = Config()

attn  = 2 * cfg.d_model * cfg.n_heads * cfg.head_dim + 2 * cfg.d_model * cfg.n_kv_heads * cfg.head_dim
ffn   = 3 * cfg.d_model * cfg.d_ff
norms = 2 * cfg.d_model + 2 * cfg.head_dim
embed = cfg.vocab_size * cfg.d_model
total = cfg.n_layers * (attn + ffn + norms) + embed + cfg.d_model

for name, v in [("attention", cfg.n_layers*attn), ("ffn", cfg.n_layers*ffn),
                ("norms", cfg.n_layers*norms + cfg.d_model), ("embedding (tied)", embed)]:
    print(f"{name:18s} {v:>12,}  {100*v/total:5.2f}%")
print(f"{'total':18s} {total:>12,}")
assert total == 596_049_920

## §2 - RMSNorm, and QK-Norm

One statistic over the feature dim, one learnable vector, no mean subtraction.

$$y = \gamma \odot \frac{x}{\sqrt{\mathrm{mean}(x^2) + \varepsilon}}$$

The same module does two jobs here: over `d_model` it normalizes the residual stream, and over
`head_dim` it becomes **QK-Norm**, applied per head to the queries and keys.

> The reference computes in float32 and casts back before applying `γ`. Skipping that cast changes the low bits.

In [ ]:
class RMSNorm(nn.Module):
    """
    y = γ * x / sqrt(mean(x²) + ε)

    Args:
        dim: Size of the normalized dimension (d_model, or head_dim for QK-Norm).
        eps: Numerical stability constant.
    """

    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        dtype = x.dtype
        x = x.to(torch.float32)                                   # compute in fp32
        x = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return self.weight * x.to(dtype)


x = torch.randn(2, 7, 64) * 5
y = RMSNorm(64)(x)
print("in  rms:", x.pow(2).mean(-1).sqrt().mean().item())
print("out rms:", y.pow(2).mean(-1).sqrt().mean().item(), "(≈1)")

## §3 - RoPE, and the convention that breaks checkpoints

Rotary embeddings rotate *pairs* of dimensions. Which dimensions form a pair is a convention,
and there are two incompatible ones:

| Convention | Pairs | Used by |
|---|---|---|
| Interleaved | `(x0, x1), (x2, x3), …` | the original RoPE paper |
| **Split-half** | `(x0, x64), (x1, x65), …` | HuggingFace, and **Qwen3 weights** |

Both are valid rotary embeddings and both train fine. A checkpoint written under one is
meaningless under the other, and nothing errors: you simply get noise.

> `rotate_half` below pairs `x[..., :64]` with `x[..., 64:]`. Pairing adjacent elements instead is the classic silent bug.

In [ ]:
def build_rope_cache(cfg: Config, seq_len: int, device, dtype):
    """
    Precompute RoPE cos/sin for positions [0, seq_len) -> two [seq_len, head_dim] tensors.

    The frequencies are duplicated because the split-half rotation pairs dimension i with
    dimension i + head_dim/2, so both halves need the same angle.
    """
    half = cfg.head_dim // 2
    inv_freq = 1.0 / (cfg.rope_theta ** (torch.arange(0, half, device=device).float() / half))
    freqs = torch.outer(torch.arange(seq_len, device=device).float(), inv_freq)  # [T, hd/2]
    emb = torch.cat((freqs, freqs), dim=-1)                                      # [T, hd]
    return emb.cos().to(dtype), emb.sin().to(dtype)


def rotate_half(x: torch.Tensor) -> torch.Tensor:
    """Split-half rotation: (x1, x2) -> (-x2, x1). NOT the interleaved variant."""
    half = x.shape[-1] // 2
    return torch.cat((-x[..., half:], x[..., :half]), dim=-1)


def apply_rope(q, k, cos, sin):
    """q, k: [B, H, T, head_dim]; cos, sin: [T, head_dim]."""
    cos, sin = cos[None, None], sin[None, None]
    return q * cos + rotate_half(q) * sin, k * cos + rotate_half(k) * sin


print("rotate_half([0,1,2,3]) =", rotate_half(torch.arange(4.)).tolist(), "-> expect [-2,-3,0,1]")

# RoPE preserves length and encodes only relative offsets
cos, sin = build_rope_cache(cfg, 8, "cpu", torch.float32)
v = torch.randn(1, 1, 8, cfg.head_dim)
r, _ = apply_rope(v, v, cos, sin)
print("norm preserved:", torch.allclose(v.norm(dim=-1), r.norm(dim=-1), atol=1e-5))

## §4 - Grouped-Query Attention with QK-Norm

Sixteen query heads share eight key/value heads, halving the KV cache. The order of operations
matters and is part of the checkpoint's contract:

$$\text{project} \;\rightarrow\; \text{QK-Norm} \;\rightarrow\; \text{RoPE} \;\rightarrow\; \text{cache} \;\rightarrow\; \text{expand KV} \;\rightarrow\; \text{attend}$$

> Each KV head must serve **consecutive** query heads: head 0 covers queries 0-1, head 1 covers 2-3. A plain tile pairs every query with the wrong keys.

In [ ]:
def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    """
    [B, n_kv, T, D] -> [B, n_kv * n_rep, T, D]

    Each KV head is repeated for CONSECUTIVE query heads.
    """
    if n_rep == 1:
        return x
    B, n_kv, T, D = x.shape
    return x[:, :, None].expand(B, n_kv, n_rep, T, D).reshape(B, n_kv * n_rep, T, D)


class KVCache:
    """Per-layer key/value store for incremental decoding."""

    def __init__(self):
        self.k = self.v = None

    def update(self, k, v):
        self.k = k if self.k is None else torch.cat([self.k, k], dim=2)
        self.v = v if self.v is None else torch.cat([self.v, v], dim=2)
        return self.k, self.v

    def __len__(self) -> int:
        return 0 if self.k is None else self.k.shape[2]


class GroupedQueryAttention(nn.Module):
    """GQA with QK-Norm and RoPE, no projection biases."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        q_out, kv_out = cfg.n_heads * cfg.head_dim, cfg.n_kv_heads * cfg.head_dim
        self.q_proj = nn.Linear(cfg.d_model, q_out, bias=False)     # 1024 -> 2048
        self.k_proj = nn.Linear(cfg.d_model, kv_out, bias=False)    # 1024 -> 1024
        self.v_proj = nn.Linear(cfg.d_model, kv_out, bias=False)
        self.o_proj = nn.Linear(q_out, cfg.d_model, bias=False)
        self.q_norm = RMSNorm(cfg.head_dim, cfg.norm_eps)           # per head, not per stream
        self.k_norm = RMSNorm(cfg.head_dim, cfg.norm_eps)

    def forward(self, x, cos, sin, cache: KVCache | None = None):
        B, T, _ = x.shape
        c = self.cfg
        q = self.q_proj(x).view(B, T, c.n_heads,    c.head_dim)     # [B, T, 16, 128]
        k = self.k_proj(x).view(B, T, c.n_kv_heads, c.head_dim)     # [B, T,  8, 128]
        v = self.v_proj(x).view(B, T, c.n_kv_heads, c.head_dim)

        q = self.q_norm(q).transpose(1, 2)                          # QK-Norm before RoPE
        k = self.k_norm(k).transpose(1, 2)
        v = v.transpose(1, 2)

        q, k = apply_rope(q, k, cos, sin)
        if cache is not None:
            k, v = cache.update(k, v)
        k, v = repeat_kv(k, c.n_rep), repeat_kv(v, c.n_rep)         # 8 -> 16 heads

        out = F.scaled_dot_product_attention(q, k, v, is_causal=(T > 1))
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))


print("repeat_kv head order:",
      repeat_kv(torch.arange(2.).view(1, 2, 1, 1).expand(1, 2, 3, 4).contiguous(), 2)[0, :, 0, 0].tolist(),
      "-> expect [0,0,1,1]")

## §5 - SwiGLU, the block, and the edges

The feed-forward network splits its up-projection in two and gates one path with SiLU:

$$\mathrm{SwiGLU}(x) = \big(\mathrm{SiLU}(x W_{\text{gate}}) \odot (x W_{\text{up}})\big) W_{\text{down}}$$

A block adds two sublayer updates into the stream; the model wraps 28 of them between an
embedding table and an LM head tied to it.

> Pre-Norm never normalizes the stream itself, so a **final norm** before the LM head is mandatory. Omitting it produces no error, only oversized logits.

In [ ]:
class SwiGLU(nn.Module):
    """y = (SiLU(x W_gate) * (x W_up)) W_down, no biases."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.up_proj   = nn.Linear(cfg.d_model, cfg.d_ff, bias=False)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))


class Block(nn.Module):
    """One pre-norm block: attention sublayer, then FFN sublayer."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.input_norm     = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn           = GroupedQueryAttention(cfg)
        self.post_attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ffn            = SwiGLU(cfg)

    def forward(self, x, cos, sin, cache=None):
        x = x + self.attn(self.input_norm(x), cos, sin, cache)      # stream stays un-normalized
        x = x + self.ffn(self.post_attn_norm(x))
        return x


class LanguageModel(nn.Module):
    """Embedding, N blocks, final norm, LM head tied to the embedding."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.embed  = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList(Block(cfg) for _ in range(cfg.n_layers))
        self.norm   = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.head   = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.head.weight = self.embed.weight                    # one matrix, two uses

    def forward(self, ids: torch.Tensor, caches=None) -> torch.Tensor:
        """ids: [B, T] -> logits [B, T, vocab_size]."""
        B, T = ids.shape
        x = self.embed(ids)
        past = len(caches[0]) if caches else 0
        cos, sin = build_rope_cache(self.cfg, past + T, ids.device, x.dtype)
        cos, sin = cos[past:], sin[past:]                            # only the new positions
        for i, block in enumerate(self.blocks):
            x = block(x, cos, sin, caches[i] if caches else None)
        return self.head(self.norm(x))


model = LanguageModel(cfg)
n = sum(p.numel() for p in {id(p): p for p in model.parameters()}.values())
print(f"built model with {n:,} parameters (formula said {total:,})")
assert n == total

## §6 - Loading the published weights

A checkpoint is a dictionary from parameter names to tensors. Loading it is a naming exercise
plus a handful of conventions that fail silently:

| Their name | Ours |
|---|---|
| `model.embed_tokens.weight` | `embed.weight` |
| `model.layers.N.input_layernorm.weight` | `blocks.N.input_norm.weight` |
| `model.layers.N.self_attn.{q,k,v,o}_proj.weight` | `blocks.N.attn.*` |
| `model.layers.N.self_attn.{q,k}_norm.weight` | `blocks.N.attn.{q,k}_norm.weight` |
| `model.layers.N.mlp.{gate,up,down}_proj.weight` | `blocks.N.ffn.*` |
| `model.norm.weight` | `norm.weight` |

> `nn.Linear` stores its weight as `[out, in]`, which is exactly how the checkpoint stores it. Transposing "to fix" the shape is a bug, not a fix.

In [ ]:
from huggingface_hub import snapshot_download
from safetensors.torch import load_file
from transformers import AutoTokenizer

path = snapshot_download(MODEL_ID, cache_dir=CACHE_DIR)
sd = load_file(os.path.join(path, "model.safetensors"))
print(f"{len(sd)} tensors in the checkpoint")
for name in list(sd)[:4]:
    print(f"  {name:52s} {tuple(sd[name].shape)}")

In [ ]:
def remap(sd: dict, cfg: Config) -> dict:
    """Map published parameter names onto our module names. Tensors are used as-is."""
    out = {"embed.weight": sd["model.embed_tokens.weight"],
           "norm.weight":  sd["model.norm.weight"]}
    for i in range(cfg.n_layers):
        src, dst = f"model.layers.{i}.", f"blocks.{i}."
        out[dst + "input_norm.weight"]     = sd[src + "input_layernorm.weight"]
        out[dst + "post_attn_norm.weight"] = sd[src + "post_attention_layernorm.weight"]
        for p in ("q_proj", "k_proj", "v_proj", "o_proj"):
            out[f"{dst}attn.{p}.weight"] = sd[f"{src}self_attn.{p}.weight"]
        for p in ("q_norm", "k_norm"):
            out[f"{dst}attn.{p}.weight"] = sd[f"{src}self_attn.{p}.weight"]
        for p in ("gate_proj", "up_proj", "down_proj"):
            out[f"{dst}ffn.{p}.weight"] = sd[f"{src}mlp.{p}.weight"]

    # tie_word_embeddings is a config flag, not something to infer from the file listing:
    # this repo happens to ship a duplicate lm_head.weight, others omit it entirely.
    out["head.weight"] = out["embed.weight"]
    return out


model.load_state_dict(remap(sd, cfg), strict=True)
model = model.to(torch.float32).eval()

tok = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)
prompt = "The CEO announced record earnings on Friday"
ids = tok(prompt, return_tensors="pt").input_ids
print("loaded ok |", ids.shape[1], "prompt tokens")

## §7 - Verifying against the reference

This is the test the whole notebook exists for. Compare logits, not generated text: sampling
hides small errors and exaggerates others.

> A difference around `1e-4` is ordinary float32 noise. A difference of `1e-1` is a bug. If it fails, hook each block and find the first layer where the residual stream diverges.

In [ ]:
from transformers import AutoModelForCausalLM

ref = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, cache_dir=CACHE_DIR, dtype=torch.float32).eval()

with torch.no_grad():
    ours, theirs = model(ids), ref(ids).logits

diff = (ours - theirs).abs().max().item()
same = torch.equal(ours.argmax(-1), theirs.argmax(-1))
print(f"max |logit diff| : {diff:.3e}")
print(f"argmax identical : {same}")
assert same and diff < 1e-2, "the implementation does not match the reference"
print("\nVERIFIED: every component matches the reference implementation")

del ref

## §8 - Generation: prefill and decode

Turning the model into text is a loop with two phases that behave very differently. Prefill
runs the whole prompt through in one pass and fills the cache; decode then runs one token per
pass, reading the entire cache each time.

> Prefill is compute-bound, decode is memory-bandwidth-bound. Without the cache, every new token would recompute keys and values for the whole prefix.

In [ ]:
@torch.no_grad()
def generate(model, ids, max_new_tokens=30, temperature=0.0, eos_id=None):
    """Prefill the prompt, then decode one token at a time through the KV cache."""
    caches = [KVCache() for _ in model.blocks]
    logits = model(ids, caches)[:, -1]                 # PREFILL: whole prompt at once

    for _ in range(max_new_tokens):
        if temperature == 0:
            nxt = logits.argmax(-1, keepdim=True)
        else:
            probs = torch.softmax(logits / temperature, dim=-1)
            nxt = torch.multinomial(probs, 1)
        ids = torch.cat([ids, nxt], dim=1)
        if eos_id is not None and nxt.item() == eos_id:
            break
        logits = model(nxt, caches)[:, -1]             # DECODE: one token per step
    return ids, caches


out, caches = generate(model, ids, max_new_tokens=30, eos_id=tok.eos_token_id)
print(repr(tok.decode(out[0], skip_special_tokens=True)))
print(f"\nKV cache after generation: {len(caches[0])} positions x {len(caches)} layers")

## What this proves

The model above was written from scratch in this notebook, yet it reproduces a published
checkpoint's logits exactly. That only happens when every convention lines up: the split-half
RoPE layout, the consecutive grouped-query expansion, QK-Norm before the rotation, the tied LM
head, and normalization on the branch rather than on the residual stream.

The one component left dense here is the feed-forward network. Replacing it with a router and
a set of experts is the subject of the next post.